# Prueba de significancia con prueba U de Mann-Whitney para resultados del baseline

## Contenido:

- Análisis de POS de los conjuntos de palabras prototípicas
- Lematización de los resultados de los algoritmos de extracción de palabras clave y de las palabras prototípicas
- Muestreo aleatorio excluyendo las STOP WORDS y lematizando las palabras
- Medidas de Jaccard por algoritmo y para las muestras aleatorias
- Prueba U de Mann-Whitney por algoritmo

## Análisis de POS para conjuntos de Palabras Prototípicas

In [1]:
# Importes

import pandas as pd
import numpy as np
import string
import spacy
import random
import os
import glob
from spacy.lang.es.stop_words import STOP_WORDS
from scipy.stats import mannwhitneyu
from collections import Counter


In [2]:
#!python -m spacy download es_core_news_sm

In [3]:
nlp = spacy.load('es_core_news_sm')
random.seed(783)

### Registro de las etiquetas de POS para palabras Prototípicas

In [4]:
#Función para obtener las etiquetas de POS y asignarlos a una nueva columna en el Data Frame

def identificaPOS(df, nombre_columna) -> pd.DataFrame:
    palabras = df[nombre_columna]

    texto=""
    for palabra in palabras:
        texto=texto+palabra+' '
    
    doc = nlp(texto)

    pos = []
    for token in doc:
        pos.append(token.pos_)
    df['POS'] = pos

    return df

In [5]:
#Adquisición de los conjuntos de palabras Prototípicas

df_asela = pd.read_csv('/home/m/MGP/maestria/proyecto_tesis/baseline/Palabras_proto/palabras_suicidio_esp.csv')
df_psiq = pd.read_csv('/home/m/MGP/maestria/proyecto_tesis/baseline/Palabras_proto/clinical-prescores.csv')
df_EmoPro = pd.read_csv('/home/m/MGP/maestria/proyecto_tesis/baseline/Palabras_proto/EmoPro/13428_2020_1519_MOESM2_ESM.csv')

In [6]:
df_EmoPro_altas = df_EmoPro[df_EmoPro['Prototypicality_Mean']>=3]

### Prueba para palabras de Asela

In [7]:
nombre = 'Palabra'
df_asela= identificaPOS(df_asela, nombre)

In [8]:
df_asela['POS'].value_counts()

POS
ADJ     36
VERB     9
NOUN     8
ADV      4
Name: count, dtype: int64

In [9]:
adverbios = [df_asela[df_asela['POS']=='ADV']]

adverbios

[    Unnamed: 0   Palabra Positiva o Negativa  pos-neg  POS
 0            0      bien            Positiva        1  ADV
 2            2       mal            Negativa       -1  ADV
 37          37     atrás            Negativa       -1  ADV
 45          45  adelante            Positiva        1  ADV]

Como se puede apreciar, las POS de las palabras prototípicas de este primer conjunto son: 
- Adjetivos
- Verbos
- Sustantivos 
- Adverbios

Con los adjetivos como la mayoría por un gran margen con 36 de 57 palabras. Los adverbios, que son la clase con menos instancias con 4, tiene antónimos de adverbios de modo: 
*mal* vs *bien* y antónimos de adverbios de lugar: *atrás* vs *adelante.*


### Prueba para palabras psiquiatras

In [10]:
nombre = 'word'
df_psiq = identificaPOS(df_psiq, nombre)

In [11]:
df_psiq['POS'].value_counts()

POS
ADJ      108
VERB      99
NOUN      92
PROPN     32
ADV        4
AUX        2
DET        1
CCONJ      1
Name: count, dtype: int64

In [12]:
adv = [df_psiq[df_psiq['POS']=='ADV']]

adv

[     Unnamed: 0       word  c-prescore  POS
 76           76  nietzsche        -4.0  ADV
 95           95  detonante        -4.0  ADV
 249         249   adelante         4.0  ADV
 332         332  tolerable         2.0  ADV]

In [13]:
aux = [df_psiq[df_psiq['POS']=='AUX']]

aux

[     Unnamed: 0   word  c-prescore  POS
 167         167    caí        -3.0  AUX
 183         183  debes        -3.0  AUX]

In [14]:
propn = [df_psiq[df_psiq['POS']=='PROPN']]

propn

[     Unnamed: 0           word  c-prescore    POS
 4             4     alprazolam        -5.0  PROPN
 5             5   escitalopram        -5.0  PROPN
 6             6        sycrest        -5.0  PROPN
 11           11        lexatin        -5.0  PROPN
 15           15         deprax        -5.0  PROPN
 16           16      lorazepam        -5.0  PROPN
 21           21       diazepam        -5.0  PROPN
 28           28        orfidal        -5.0  PROPN
 29           29       pastilla        -5.0  PROPN
 30           30         liryca        -5.0  PROPN
 44           44     esclerosis        -4.0  PROPN
 58           58         azotea        -4.0  PROPN
 64           64  desistimiento        -4.0  PROPN
 70           70   intolerancia        -4.0  PROPN
 88           88       alboroto        -4.0  PROPN
 91           91           bebí        -4.0  PROPN
 118         118      desconfío        -4.0  PROPN
 141         141         muerde        -3.0  PROPN
 153         153       aparento

In [15]:
det = [df_psiq[df_psiq['POS']=='DET']]

det

[   Unnamed: 0 word  c-prescore  POS
 1           1   mg        -5.0  DET]

In [16]:
cconj = [df_psiq[df_psiq['POS']=='CCONJ']]

cconj

[     Unnamed: 0   word  c-prescore    POS
 125         125  sopor        -3.0  CCONJ]

A pesar de que en este caso se identifican 8 diferentes POS, algunas de las etiquetas minoritarias parecen estar mal, algunos ejemplos no corresponden a la etiqueta indicada

## Cálculo de Jaccard con lematización

### Recolección de los resultados de los algoritmos y del dataset para recuperar el vocabulario

#### Funciones

In [17]:
#Función para obtener las palabras lematizadas y asignarlas a una nueva columna en el Data Frame

def lematizador(df, nombre_columna) -> pd.DataFrame:
    palabras = df[nombre_columna]

    texto=""
    for palabra in palabras:
        texto=texto+palabra+' '
    
    doc = nlp(texto)
    if len(doc) >len(palabras):
        doc = doc[:len(palabras)]
    lem = []
    for token in doc:
        lem.append(token.lemma_)
    df['Lemma'] = lem

    return df

In [18]:
#Función para recuperar el vocabulario y el texto sin stop words como lista

def cargar_datos(filename) -> list:
    textopos = []
    textoneg = []
    texto0 = []
    texto5 = []
    with open(filename, encoding="utf-8-sig") as f:
        df = pd.read_csv(f)
        texto = df['Texto']
        for _, r in df.iterrows():
            if r['ds0':'ds3'].sum(axis=0) == 1:
                textoneg.append(r['Texto'])
            else:
                textopos.append(r['Texto'])
            if r['ds0'] == 1:
                texto0.append(r['Texto'])
            elif r['ds5'] == 1:
                texto5.append(r['Texto'])
        #pasamos a minúsculas
        for row in texto:
            row = ' '.join(row)
            row = row.lower()
        for row in textopos:
            row = ' '.join(row)
            row = row.lower()
        for row in textoneg:
            row = ' '.join(row)
            row = row.lower()
        for row in texto0:
            row = ' '.join(row)
            row = row.lower()
        for row in texto5:
            row = ' '.join(row)
            row = row.lower()
        
    #Eliminar puntuación
    texto =' '.join(texto)
    textopos =' '.join(textopos)
    textoneg =' '.join(textoneg)
    texto0 =' '.join(texto0)
    texto5 =' '.join(texto5)
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = texto.translate(str.maketrans('', '', '¿?!¡—“”0123456789][’'))
    textopos = textopos.translate(str.maketrans('', '', string.punctuation))
    textopos = textopos.translate(str.maketrans('', '', '¿?!¡—“”0123456789][’'))
    textoneg = textoneg.translate(str.maketrans('', '', string.punctuation))
    textoneg = textoneg.translate(str.maketrans('', '', '¿?!¡—“”0123456789][’'))
    texto0 = texto0.translate(str.maketrans('', '', string.punctuation))
    texto0 = texto0.translate(str.maketrans('', '', '¿?!¡—“”0123456789][’'))
    texto5 = texto5.translate(str.maketrans('', '', string.punctuation))
    texto5 = texto5.translate(str.maketrans('', '', '¿?!¡—“”0123456789][’'))
    palabras = texto.split()
    palabraspos = textopos.split()
    palabrasneg = textoneg.split()
    palabras0 = texto0.split()
    palabras5 = texto5.split()
    #rearmamos el texto debido a que existen carácteres especiales
    texto_palabras = ""
    texto_pos = ""
    texto_neg = ""
    texto_5 = ""
    texto_0 = ""
    for palabra in palabras:
        if palabra not in STOP_WORDS:
            texto_palabras=texto_palabras+palabra+" "
    tokens = texto_palabras.split()


    for palabra in palabraspos:
        if palabra not in STOP_WORDS:
            texto_pos=texto_pos+palabra+" "
    tokens_pos = texto_pos.split()


    for palabra in palabrasneg:
        if palabra not in STOP_WORDS:
            texto_neg=texto_neg+palabra+" "
    tokens_neg = texto_neg.split()

    
    for palabra in palabras0:
        if palabra not in STOP_WORDS:
            texto_0=texto_0+palabra+" "
    tokens_0 = texto_0.split()

    for palabra in palabras5:
        if palabra not in STOP_WORDS:
            texto_5=texto_5+palabra+" "
    tokens_5 = texto_5.split()

    return tokens, tokens_pos, tokens_neg, tokens_5, tokens_0
    

In [19]:
#Función para la selección aleatoria
def randomSelect(size, vocab) -> list:
    select = []
    while(size != len(select)):
        x = random.randint(0, len(vocab)-1)
        if vocab[x] not in select:
            select.append(vocab[x])
    return select

In [20]:
#Función para la medida de jaccard
def jaccard(s1: set, s2: set) -> float:
    return len(s1 & s2) / len(s1 | s2)   

In [21]:
# Función para obtener la lista de puntuaciones Jaccard por algoritmo

def listaJaccard(dict_algoritmos, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas) -> list:
    puntuaciones = []
    for k, v in dict_algoritmos.items():
        if 'asela' in k:
            puntuaciones.append((k, jaccard(set(df_asela['Lemma']), set(v['Lemma']))))
        elif'psiq' in k:
            puntuaciones.append((k, jaccard(set(v['Lemma']), set(df_psiq['Lemma']))))
        elif 'alta' in k:
            puntuaciones.append((k, jaccard(set(v['Lemma']), set(df_EmoPro_altas['Lemma']))))
        else:
            puntuaciones.append((k, jaccard(set(v['Lemma']), set(df_EmoPro['Lemma']))))
    return puntuaciones



In [22]:
#Función para generar los diccionarios con los DF con las palabras y añadir una columna con los lemas
def lectorResultados(lista_documentos, path, columna) -> pd.DataFrame:
    data_frames = {}
    for documento in lista_documentos:
        nombre = os.path.splitext(os.path.basename(documento))[0]
        data_frames[nombre] = pd.read_csv(path+documento)
    for k,v in data_frames.items():
        k = lematizador(v, columna)
    return data_frames

In [23]:
# paths para los diferentes conjuntos de eresultados de los diferentes algoritmos

path_fractalidad = '/home/m/MGP/maestria/proyecto_tesis/baseline/output/Fractalidad/'

path_TextRank = '/home/m/MGP/maestria/proyecto_tesis/baseline/output/TextRank/'

path_GPT = '/home/m/MGP/maestria/proyecto_tesis/baseline/output/LLMs/GPT/'

path_KeyBERT = '/home/m/MGP/maestria/proyecto_tesis/baseline/output/KeyBERT/'

#### Conjuntos de palabras obtenidas por algoritmos

In [24]:
#Lista de archivos con resultados de fractalidad

resultados_fractalidad = [archivo for archivo in glob.glob('fractalidad*.csv', root_dir=path_fractalidad)]

resultados_fractalidad

['fractalidad_etiqueta0_asela.csv',
 'fractalidad_todos_EmoPro.csv',
 'fractalidad_negativas_resultados_asela.csv',
 'fractalidad_etiqueta5_EmoPro_prueba1.csv',
 'fractalidad_etiqueta0_EmoPro_prueba1.csv',
 'fractalidad_etiqueta5_asela.csv',
 'fractalidad_polaridad_neg_EmoPro.csv',
 'fractalidad_etiqueta0_psiq.csv',
 'fractalidad_polaridad_pos_EmoPro.csv',
 'fractalidad_todos_psiq.csv',
 'fractalidad_resultados_todas_asela.csv',
 'fractalidad_positivas_polaridad_psiq.csv',
 'fractalidad_positivas_resultados_asela.csv',
 'fractalidad_etiqueta5_psiq.csv',
 'fractalidad_negativas_polaridad_psiq.csv']

In [25]:
#Lista de archivos con resultados de TextRank

resultados_TextRank = [archivo for archivo in glob.glob('Corregido*.csv', root_dir=path_TextRank)]

resultados_TextRank

['Corregido_TextRank_polaridad_negativas_EmoPro.csv',
 'Corregido_TextRank_etiqueta_negativas_EmoPro_alta.csv',
 'Corregido_TextRank_etiqueta_negativas_psiq.csv',
 'Corregido_TextRank_etiqueta_positivas_EmoPro.csv',
 'Corregido_TextRank_polaridad_positivas_EmoPro.csv',
 'Corregido_TextRank_polaridad_negativas_EmoPro_alta.csv',
 'Corregido_TextRank_polaridad_negativas_psiq.csv',
 'Corregido_TextRank_todas _EmoPro_alta.csv',
 'Corregido_TextRank_etiqueta_positivas_EmoPro_alta.csv',
 'Corregido_TextRank_polaridad_positivas_EmoPro_alta.csv',
 'Corregido_TextRank_todas_psiq.csv',
 'Corregido_TextRank_todas_EmoPro.csv',
 'Corregido_TextRank_etiqueta_negativas_EmoPro.csv',
 'Corregido_TextRank_polaridad_positivas_psiq.csv',
 'Corregido_TextRank_etiqueta_positivas_psiq.csv']

In [26]:
#Lista de archivos con resultados de GPT

resultados_GPT = [archivo for archivo in glob.glob('palabras*.csv', root_dir=path_GPT)]

resultados_GPT

['palabras_prototipicas_100_EmoPro.csv',
 'palabras_emocionales_GPT_EmoPro_alta.csv',
 'palabras_emocionales_prompt_proto_EmoPro_alta.csv',
 'palabras_emocionales_GPT_psiq.csv',
 'palabras_emocionales_prompt_proto_psiq.csv',
 'palabras_prototipicas_100_asela.csv',
 'palabras_emocionales_prompt_proto_asela.csv',
 'palabras_prototipicas_100_EmoPro_alta.csv',
 'palabras_resultados_GPT_textos_unidos_EmoPro_alta.csv',
 'palabras_emocionales_prompt_proto_EmoPro.csv',
 'palabras_resultados_GPT_textos_unidos_psiq.csv',
 'palabras_emocionales_GPT_EmoPro.csv',
 'palabras_emocionales_GPT_asela.csv',
 'palabras_resultados_GPT_textos_unidos_asela.csv',
 'palabras_resultados_GPT_textos_unidos_EmoPro.csv',
 'palabras_prototipicas_100_psiq.csv']

In [27]:
#Lista de archivos con resultados de KeyBERT

resultados_KeyBERT = [archivo for archivo in glob.glob('Salida*.txt', root_dir=path_KeyBERT)]

resultados_KeyBERT

['Salida_Etiqueta0_KB_asela.txt',
 'Salida_Negativas_Todos_KB_asela.txt',
 'Salida_Etiqueta0_KB_psiq.txt',
 'Salida_Todos_KB_asela.txt',
 'Salida_Todos_KB_psiq.txt',
 'Salida_UnoporUno_KB_EmoPro.txt',
 'Salida_Etiqueta5_KB_psiq.txt',
 'Salida_Positivas_KB_psiq.txt',
 'Salida_Negativas_KB_EmoPro.txt',
 'Salida_UnoporUno_KB_final_psiq.txt',
 'Salida_Etiqueta5_KB_EmoPro.txt',
 'Salida_Etiqueta5_KB_asela.txt',
 'Salida_Positivas_KB_EmoPro.txt',
 'Salida_Positivas_Todos_KB_asela.txt',
 'Salida_Etiqueta0_KB_EmoPro.txt',
 'Salida_KB_Todos_EmoPro.txt',
 'Salida_Negativas_KB_psiq.txt',
 'Salida_UnoporUno_KB_final_asela.txt']

In [28]:

"""
Celda para crear el diccionario con los DataFrames de los resultados y lematización de KeyBERT porque
los resultados están guardados en formato txt y la función no va a funcionar con el texto de entrada

"""

df_resultados_KeyBERT = {}
for documento in resultados_KeyBERT:
    nombre = os.path.splitext(os.path.basename(documento))[0]
    with open(path_KeyBERT+documento, 'r') as f:
        df_resultados_KeyBERT[nombre] = f.readlines()
df_resultados_KeyBERT

for k,v in df_resultados_KeyBERT.items():    
    texto =""
    for palabra in df_resultados_KeyBERT[k]:
        texto =texto+palabra+' '

    texto= texto.strip()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = texto.translate(str.maketrans('', '', '¿?!¡—“”0123456789][’'))
    texto = ''.join(texto)

        

    doc = nlp(texto)


    data = {'word':[], 'Lemma':[]}
    for token in doc:
        data['word'].append(token)
        data['Lemma'].append(token.lemma_)

    df_resultados_KeyBERT[k] = pd.DataFrame(data = data)



In [29]:
# Lectura de datos y obtención de lemas para hacer las mediciones de Jaccard
df_resultados_TextRank = lectorResultados(resultados_TextRank, path_TextRank, 'word')
df_resultados_GPT = lectorResultados(resultados_GPT, path_GPT, 'palabra')
df_resultados_fractalidad = lectorResultados(resultados_fractalidad, path_fractalidad, 'word')


# Creación de DataFrames con 57 resultados para comparar con las palabras de Asela
df_resultados_TextRank['Corregido_TextRank_polaridad_negativas_asela'] = df_resultados_TextRank['Corregido_TextRank_polaridad_negativas_psiq'][:57]
df_resultados_TextRank['Corregido_TextRank_todas_asela'] = df_resultados_TextRank['Corregido_TextRank_todas_psiq'][:57]
df_resultados_TextRank['Corregido_TextRank_polaridad_positivas_asela'] = df_resultados_TextRank['Corregido_TextRank_polaridad_positivas_psiq'][:57]
df_resultados_TextRank['Corregido_TextRank_etiqueta_positivas_asela'] = df_resultados_TextRank['Corregido_TextRank_etiqueta_positivas_psiq'][:57]
df_resultados_TextRank['Corregido_TextRank_etiqueta_negativas_asela'] = df_resultados_TextRank['Corregido_TextRank_etiqueta_negativas_psiq'][:57]

In [30]:
# Creación de DataFrames para hacer la comparación con las altamente prototípicas para Fractalidad
df_resultados_fractalidad['fractalidad_todos_EmoPro_altas'] = df_resultados_fractalidad['fractalidad_todos_EmoPro'][:549]
df_resultados_fractalidad['fractalidad_polaridad_pos_EmoPro_altas'] = df_resultados_fractalidad['fractalidad_polaridad_pos_EmoPro'][:549]
df_resultados_fractalidad['fractalidad_polaridad_neg_EmoPro_altas'] = df_resultados_fractalidad['fractalidad_polaridad_neg_EmoPro'][:549]
df_resultados_fractalidad['fractalidad_etiqueta5_EmoPro_prueba1_altas'] = df_resultados_fractalidad['fractalidad_etiqueta5_EmoPro_prueba1'][:549]
df_resultados_fractalidad['fractalidad_etiqueta0_EmoPro_prueba1_altas'] = df_resultados_fractalidad['fractalidad_etiqueta0_EmoPro_prueba1'][:549]


# Creación de DataFrames para hacer la comparación con las altamente prototípicas KeyBERT
df_resultados_KeyBERT['Salida_UnoporUno_KB_EmoPro_altas'] = df_resultados_KeyBERT['Salida_UnoporUno_KB_EmoPro'][:549]
df_resultados_KeyBERT['Salida_Negativas_KB_EmoPro_altas'] = df_resultados_KeyBERT['Salida_Negativas_KB_EmoPro'][:549]
df_resultados_KeyBERT['Salida_Etiqueta5_KB_EmoPro_altas'] = df_resultados_KeyBERT['Salida_Etiqueta5_KB_EmoPro'][:549]
df_resultados_KeyBERT['Salida_Positivas_KB_EmoPro_altas'] = df_resultados_KeyBERT['Salida_Positivas_KB_EmoPro'][:549]
df_resultados_KeyBERT['Salida_Etiqueta0_KB_EmoPro_altas'] = df_resultados_KeyBERT['Salida_Etiqueta0_KB_EmoPro'][:549]
df_resultados_KeyBERT['Salida_KB_Todos_EmoPro_altas'] = df_resultados_KeyBERT['Salida_KB_Todos_EmoPro'][:549]

In [31]:
# Lematización de las palabras prototípicas

df_asela = lematizador(df_asela, 'Palabra')
df_EmoPro = lematizador(df_EmoPro, 'Word')
df_EmoPro_altas = lematizador(df_EmoPro_altas, 'Word')
df_psiq = lematizador(df_psiq, 'word')

### Cálculo de medidas de Jaccard por algoritmo para todos los conjuntos de palabras

In [32]:
conjunto_jaccard_TextRanK = listaJaccard(df_resultados_TextRank, df_asela= df_asela, df_EmoPro= df_EmoPro, df_EmoPro_altas= df_EmoPro_altas, df_psiq=df_psiq)
conjunto_jaccard_Fractalidad = listaJaccard(df_resultados_fractalidad, df_asela= df_asela, df_EmoPro= df_EmoPro, df_EmoPro_altas= df_EmoPro_altas, df_psiq=df_psiq)
conjunto_jaccard_KeyBERT = listaJaccard(df_resultados_KeyBERT, df_asela= df_asela, df_EmoPro= df_EmoPro, df_EmoPro_altas= df_EmoPro_altas, df_psiq=df_psiq)
conjunto_jaccard_GPT = listaJaccard(df_resultados_GPT, df_asela= df_asela, df_EmoPro= df_EmoPro, df_EmoPro_altas= df_EmoPro_altas, df_psiq=df_psiq)


In [33]:
print(len(conjunto_jaccard_Fractalidad))
print(len(conjunto_jaccard_GPT))
print(len(conjunto_jaccard_KeyBERT))
print(len(conjunto_jaccard_TextRanK))

20
16
24
20


In [34]:
texto, textopos, textoneg, texto5, texto0 = cargar_datos('/home/m/MGP/maestria/proyecto_tesis/baseline/Datasets/SMS_DATA_ORIGINAL.csv')

In [35]:
print(len(texto))
print(len(textopos))
print(len(textoneg))
print(len(texto5))
print(len(texto0))

83388
50270
33118
34828
8199


In [36]:
texto5[0]

'mes'

## Generación de muestras aleatorias y cálculo de medidas de Jaccard

### Muestreo aleatorio con Jaccard por algoritmo

In [37]:
# Función para calcular la métrica de Jaccard para las muestras aleatorias

def randomJaccard(vocabulario, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas) -> list:
    y = []
    muestra = pd.DataFrame(randomSelect(len(df_asela), vocabulario), columns=['word'])
    muestra = lematizador(muestra, 'word')
    puntaje = jaccard(set(muestra['Lemma']), set(df_asela['Lemma']))
    y.append(puntaje)

    muestra = pd.DataFrame(randomSelect(len(df_psiq), vocabulario), columns=['word'])
    muestra = lematizador(muestra, 'word')
    puntaje = jaccard(set(muestra['Lemma']), set(df_psiq['Lemma']))
    y.append(puntaje)

    muestra = pd.DataFrame(randomSelect(len(df_EmoPro), vocabulario), columns=['word'])
    muestra = lematizador(muestra, 'word')
    puntaje = jaccard(set(muestra['Lemma']), set(df_EmoPro['Lemma']))
    y.append(puntaje)

    muestra = pd.DataFrame(randomSelect(len(df_EmoPro_altas), vocabulario), columns=['word'])
    muestra = lematizador(muestra, 'word')
    puntaje = jaccard(set(muestra['Lemma']), set(df_EmoPro_altas['Lemma']))
    y.append(puntaje)
    return y


In [38]:
#Conjunto de listas con índice de Jaccard para comparación con TextRank agrupados  por modo de extracción

y_TextRank_texto=randomJaccard(texto, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_TextRank_pos=randomJaccard(textopos, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_TextRank_neg=randomJaccard(textoneg, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_TextRank_5=randomJaccard(texto5, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_TextRank_0=randomJaccard(texto0, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)

#Conjunto de listas con índice de Jaccard para comparación con Fractalidad agrupados  por modo de extracción
y_Fractalidad_texto=randomJaccard(texto, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_Fractalidad_pos=randomJaccard(textopos, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_Fractalidad_neg=randomJaccard(textoneg, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_Fractalidad_5=randomJaccard(texto5, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_Fractalidad_0=randomJaccard(texto0, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)

#Conjunto de listas con índice de Jaccard para comparación con KeyBERT agrupados  por modo de extracción
y_KeyBERT_texto=randomJaccard(texto, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_KeyBERT_pos=randomJaccard(textopos, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_KeyBERT_neg=randomJaccard(textoneg, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_KeyBERT_5=randomJaccard(texto5, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_KeyBERT_0=randomJaccard(texto0, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
y_KeyBERT_unoauno=randomJaccard(texto, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)

#Conjunto de listas con índice de Jaccard para comparación con GPT agrupados  por modo de extracción
y_GPT_texto= []
for _ in range(4):
    y=randomJaccard(texto, df_asela, df_psiq, df_EmoPro, df_EmoPro_altas)
    for i in range(len(y)):
        y_GPT_texto.append(i)


In [39]:
conjunto_jaccard_Fractalidad

[('fractalidad_etiqueta0_asela', 0.009523809523809525),
 ('fractalidad_todos_EmoPro', 0.041631973355537054),
 ('fractalidad_negativas_resultados_asela', 0.0),
 ('fractalidad_etiqueta5_EmoPro_prueba1', 0.05140388768898488),
 ('fractalidad_etiqueta0_EmoPro_prueba1', 0.05187319884726225),
 ('fractalidad_etiqueta5_asela', 0.0),
 ('fractalidad_polaridad_neg_EmoPro', 0.051360842844600525),
 ('fractalidad_etiqueta0_psiq', 0.050488599348534204),
 ('fractalidad_polaridad_pos_EmoPro', 0.04759881002974926),
 ('fractalidad_todos_psiq', 0.015267175572519083),
 ('fractalidad_resultados_todas_asela', 0.0),
 ('fractalidad_positivas_polaridad_psiq', 0.024691358024691357),
 ('fractalidad_positivas_resultados_asela', 0.0),
 ('fractalidad_etiqueta5_psiq', 0.029503105590062112),
 ('fractalidad_negativas_polaridad_psiq', 0.027906976744186046),
 ('fractalidad_todos_EmoPro_altas', 0.029551954242135366),
 ('fractalidad_polaridad_pos_EmoPro_altas', 0.02995169082125604),
 ('fractalidad_polaridad_neg_EmoPro_altas

In [40]:
conjunto_jaccard_GPT

[('palabras_prototipicas_100_EmoPro', 0.018642803877703208),
 ('palabras_emocionales_GPT_EmoPro_alta', 0.0782312925170068),
 ('palabras_emocionales_prompt_proto_EmoPro_alta', 0.10207612456747404),
 ('palabras_emocionales_GPT_psiq', 0.02444987775061125),
 ('palabras_emocionales_prompt_proto_psiq', 0.009569377990430622),
 ('palabras_prototipicas_100_asela', 0.007246376811594203),
 ('palabras_emocionales_prompt_proto_asela', 0.0),
 ('palabras_prototipicas_100_EmoPro_alta', 0.024154589371980676),
 ('palabras_resultados_GPT_textos_unidos_EmoPro_alta', 0.023809523809523808),
 ('palabras_emocionales_prompt_proto_EmoPro', 0.05723124516627997),
 ('palabras_resultados_GPT_textos_unidos_psiq', 0.005194805194805195),
 ('palabras_emocionales_GPT_EmoPro', 0.047619047619047616),
 ('palabras_emocionales_GPT_asela', 0.06201550387596899),
 ('palabras_resultados_GPT_textos_unidos_asela', 0.0),
 ('palabras_resultados_GPT_textos_unidos_EmoPro', 0.01524390243902439),
 ('palabras_prototipicas_100_psiq', 0.00

In [48]:
#Conjunto de índices Jaccard de los algoritmos por agrupamiento de texto
#TextRank
x_TextRank_todas = [x[1] for x in conjunto_jaccard_TextRanK if 'todas' in x[0]]
x_TextRank_pos = [x[1] for x in conjunto_jaccard_TextRanK if 'polaridad_positiva' in x[0]]
x_TextRank_neg = [x[1] for x in conjunto_jaccard_TextRanK if 'polaridad_negativa' in x[0]]
x_TextRank_0 = [x[1] for x in conjunto_jaccard_TextRanK if 'etiqueta_negativa' in x[0]]
x_TextRank_5 = [x[1] for x in conjunto_jaccard_TextRanK if 'etiqueta_positiva' in x[0]]

#Fractalidad
x_Fractalidad_todas = [x[1] for x in conjunto_jaccard_Fractalidad if 'todas' in x[0]]
x_Fractalidad_pos = [x[1] for x in conjunto_jaccard_Fractalidad if 'pos' in x[0]]
x_Fractalidad_neg = [x[1] for x in conjunto_jaccard_Fractalidad if 'neg' in x[0]]
x_Fractalidad_0 = [x[1] for x in conjunto_jaccard_Fractalidad if '0' in x[0]]
x_Fractalidad_5 = [x[1] for x in conjunto_jaccard_Fractalidad if '5' in x[0]]

#KeyBERT
x_KeyBERT_todas = [x[1] for x in conjunto_jaccard_KeyBERT if 'Todos' in x[0]]
x_KeyBERT_pos = [x[1] for x in conjunto_jaccard_KeyBERT if 'Positivas' in x[0]]
x_KeyBERT_neg = [x[1] for x in conjunto_jaccard_KeyBERT if 'Negativas' in x[0]]
x_KeyBERT_0 = [x[1] for x in conjunto_jaccard_KeyBERT if '0' in x[0]]
x_KeyBERT_5 = [x[1] for x in conjunto_jaccard_KeyBERT if '5' in x[0]]
x_KeyBERT_Unoporuno = [x[1] for x in conjunto_jaccard_KeyBERT if 'UnoporUno' in x[0]]

#GPT
x_GPT = [x[1] for x in conjunto_jaccard_GPT]

In [49]:
x_KeyBERT_todas

[0.0,
 0.0,
 0.021638330757341576,
 0.009708737864077669,
 0.029278350515463916,
 0.022792022792022793]

## Prueba de Mann-Whitney

La prueba U de Mann-Whitney es una prueba estadística que busca identificar si hay diferencia entre dos muestras. Es la alternativa no paramétrica de la prueba t-student. A diferencia de ésta, mide la diferencia, no de los promedios de las muestras, sino de la suma de rankings.
El procedimiento es el siguiente:
1. Ordenar de froma ascendente todos los valores de ambas muestras y asignarles un ranking de acuerdo a la posición del ordenamiento
2. Sumar los valores de los rankings para cada grupo: 

    $T_{1}=\sum_{i=1}^{k}t_{i}$  

    $T_{2}=\sum_{i=1}^{k}t_{i}$ 

    donde $T_{i}$ es el ranking de cada elemento

3. Calcular $U$: 

    $U_{1}=n_{1}n_{2} + \frac{n_{1}(n_{1}+1)}{2} - T_{1}$

    $U_{2}=n_{2}n_{1}+\frac{n_{2}(n_{2}+1)}{2} - T_{2}$

    donde $n_{i}$ es el número de instancias del grupo $i$
    La $U$ de la pruebá será $U= min(U_{1}, U_{2})$

4. Calcular la $U$ esperada: $\mu U = \frac{n_{1}n_{2}}{2}$

5. Calcular la desviación estandar de $U$: $\sigma U = \sqrt{\frac{n_{1}n_{2}(n_{1}+n_{2}+1)}{12}}$

6. Calculamos el valor estandarizado $U$: $z=\frac{U-\mu U}{\sigma U}$

**Hipótesis** de la prueba:
- $h_{0}:$ En ambas muestras la suma de rankings no difiere de forma significativa.
- $h_{1}:$ En ambas muestras la suma de rankings difiere significativamente

In [50]:
#p-value de conjuntos de TextRank
U_TRt, p_value_TR_t = mannwhitneyu(x_TextRank_todas, y_TextRank_texto)
U_TRp, p_value_TR_p = mannwhitneyu(x_TextRank_pos, y_TextRank_pos)
U_TRn, p_value_TR_n = mannwhitneyu(x_TextRank_neg, y_TextRank_neg)
U_TR5, p_value_TR_5 = mannwhitneyu(x_TextRank_5, y_TextRank_5)
U_TR0, p_value_TR_0 = mannwhitneyu(x_TextRank_0, y_TextRank_0)

#p-value de conjuntos de Fractalidad
U_Frt, p_value_Fr_t = mannwhitneyu(x_Fractalidad_todas, y_Fractalidad_texto)
U_Frp, p_value_Fr_p = mannwhitneyu(x_Fractalidad_pos, y_Fractalidad_pos)
U_Frn, p_value_Fr_n = mannwhitneyu(x_Fractalidad_neg, y_Fractalidad_neg)
U_Fr5, p_value_Fr_5 = mannwhitneyu(x_Fractalidad_5, y_Fractalidad_5)
U_Fr0, p_value_Fr_0 = mannwhitneyu(x_Fractalidad_0, y_Fractalidad_0)

#p-value de conjunto de KeyBERT
U_KB_t, p_value_KB_t = mannwhitneyu(x_KeyBERT_todas, y_KeyBERT_texto)
U_KB_p, p_value_KB_p = mannwhitneyu(x_KeyBERT_pos, y_KeyBERT_pos)
U_KB_n, p_value_KB_n = mannwhitneyu(x_KeyBERT_neg, y_KeyBERT_neg)
U_KB_5, p_value_KB_5 = mannwhitneyu(x_KeyBERT_5, y_KeyBERT_5)
U_KB_0, p_value_KB_0 = mannwhitneyu(x_KeyBERT_0, y_KeyBERT_0)
U_KB_Unoauno, p_value_KB_Unoauno = mannwhitneyu(x_KeyBERT_Unoporuno, y_KeyBERT_unoauno)

#p-value para conjunto de GPT
U_GPT_t, p_value_GPT_t = mannwhitneyu(x_GPT, y_GPT_texto)

In [52]:
p_value_KB_t

np.float64(0.10874205375165716)

In [53]:
print('p-value de TextRank vs texto completo: ', p_value_TR_t)
print('p-value de TextRank vs texto positivo: ', p_value_TR_p)
print('p-value de TextRank vs texto negativo: ', p_value_TR_n)
print('p-value de TextRank vs texto etiqueta 5: ', p_value_TR_5)
print('p-value de TextRank vs texto etiqueta 0: ', p_value_TR_0)


print('p-value de Fractalidad vs texto completo: ', p_value_Fr_t)
print('p-value de Fractalidad vs texto positivo: ', p_value_Fr_p)
print('p-value de Fractalidad vs texto negativo: ', p_value_Fr_n)
print('p-value de Fractalidad vs texto etiqueta 5: ', p_value_Fr_5)
print('p-value de Fractalidad vs texto etiqueta 0: ', p_value_Fr_0)


print('p-value de KeyBERT vs texto completo: ', p_value_KB_t)
print('p-value de KeyBERT vs texto positivo: ', p_value_KB_p)
print('p-value de KeyBERT vs texto negativo: ', p_value_KB_n)
print('p-value de KeyBERT vs texto etiqueta 5: ', p_value_KB_5)
print('p-value de KeyBERT vs texto etiqueta 0: ', p_value_KB_0)
print('p-value de KeyBERT vs texto Uno por uno: ', p_value_KB_Unoauno)


print('p-value de GPT vs texto: ', p_value_GPT_t)

p-value de TextRank vs texto completo:  0.4857142857142857
p-value de TextRank vs texto positivo:  0.34285714285714286
p-value de TextRank vs texto negativo:  0.4857142857142857
p-value de TextRank vs texto etiqueta 5:  0.6857142857142857
p-value de TextRank vs texto etiqueta 0:  0.4857142857142857
p-value de Fractalidad vs texto completo:  0.4
p-value de Fractalidad vs texto positivo:  0.2
p-value de Fractalidad vs texto negativo:  0.4857142857142857
p-value de Fractalidad vs texto etiqueta 5:  0.4857142857142857
p-value de Fractalidad vs texto etiqueta 0:  0.8857142857142857
p-value de KeyBERT vs texto completo:  0.10874205375165716
p-value de KeyBERT vs texto positivo:  0.11428571428571428
p-value de KeyBERT vs texto negativo:  0.11428571428571428
p-value de KeyBERT vs texto etiqueta 5:  0.34285714285714286
p-value de KeyBERT vs texto etiqueta 0:  0.8857142857142857
p-value de KeyBERT vs texto Uno por uno:  0.05714285714285714
p-value de GPT vs texto:  0.010488538912201287


In [48]:
p_values = {'p-value_TR':[p_value_TR_t], 'p-value_Frac': [p_value_Fr_t], 'p-value_KB': [p_value_KB_t], 'p-value_GPT': [p_value_GPT_t]}

pv_final = pd.DataFrame(p_values)

pv_final

,p-value_TR,p-value_Frac,p-value_KB,p-value_GPT
0,0.107445,0.107313,0.020224,0.100991


In [49]:
pv_final = pv_final.T
# pv_final.to_csv('p-values_Mann-Whitney.csv')

,0
p-value_TR,0.107445
p-value_Frac,0.107313
p-value_KB,0.020224
p-value_GPT,0.100991


### Pendientes
Falta la comparación directa entre el contenido de los algoritmos, el azar y las top_n palabras más frecuentes

## Comparaciones de Jaccard entre top_n palabras frecuentes, prototípicas, algoritmos y azar

### Jaccard entre top_n por frecuencia y palabras proto

In [45]:
"""
Las siguientes celdas de código tienen el propósito de armar una lista con las palabras del vocabulario ordenadas por
su frecuencia de forna descendente. Esto con el fin de medir el índice Jaccard entre esta lista y los conjuntos de palabras prototípicas
"""
conteos_tokens = Counter(texto)

cont_ordenados=[(k,v) for (k,v) in conteos_tokens.items()]
cont_ordenados = sorted(cont_ordenados, key =lambda tupla: tupla[1], reverse = True)
cont_ordenados = cont_ordenados[:len(df_EmoPro)]

textop = ""
for palabra in cont_ordenados:
    textop = textop+palabra[0]+' '
textop = ''.join(textop)
doc = nlp(textop)

data = {'word':[], 'Lemma':[]}
for token in doc:
    data['word'].append(token)
    data['Lemma'].append(token.lemma_)

df_palabras_frecuencia = pd.DataFrame(data= data)

In [46]:
#Obtención del índice de Jaccard entre top_n y palabras prototípicas de cada conjunto

jaccard_frecuencia_asela = jaccard(set(df_asela['Lemma']), set(df_palabras_frecuencia['Lemma'][:len(df_asela)]))
jaccard_frecuencia_psiq = jaccard(set(df_psiq['Lemma']), set(df_palabras_frecuencia['Lemma'][:len(df_psiq)]))
jaccard_frecuencia_EmoPro_altas = jaccard(set(df_EmoPro_altas['Lemma']), set(df_palabras_frecuencia['Lemma'][:len(df_EmoPro_altas)]))
jaccard_frecuencia_EmoPro = jaccard(set(df_EmoPro['Lemma']), set(df_palabras_frecuencia['Lemma']))


In [47]:
print("Jaccard entre más frecuentes y Asela: ", jaccard_frecuencia_asela)
print("Jaccard entre más frecuentes y psiquiatras: ", jaccard_frecuencia_psiq)
print("Jaccard entre más frecuentes y EmoPro altamente protos: ", jaccard_frecuencia_EmoPro_altas)
print("Jaccard entre más frecuentes y EmoPro: ", jaccard_frecuencia_EmoPro)

Jaccard entre más frecuentes y Asela:  0.07216494845360824
Jaccard entre más frecuentes y psiquiatras:  0.04377104377104377
Jaccard entre más frecuentes y EmoPro altamente protos:  0.04573804573804574
Jaccard entre más frecuentes y EmoPro:  0.05958429561200924


### Medida de Jaccard entre top_n y algoritmos

In [48]:
# Obtención de índice de Jaccard entre top_n y resultados de algoritmos usando la función de listaJaccard
puntuaciones_textRank_topn = listaJaccard(df_resultados_TextRank, df_asela=df_palabras_frecuencia[:len(df_asela)], df_psiq=df_palabras_frecuencia[:len(df_psiq)], df_EmoPro=df_palabras_frecuencia, df_EmoPro_altas=df_palabras_frecuencia[:len(df_EmoPro_altas)])
puntuaciones_fractalidad_topn = listaJaccard(df_resultados_fractalidad, df_asela=df_palabras_frecuencia[:len(df_asela)], df_psiq=df_palabras_frecuencia[:len(df_psiq)], df_EmoPro=df_palabras_frecuencia, df_EmoPro_altas=df_palabras_frecuencia[:len(df_EmoPro_altas)])
puntuaciones_KeyBERT_topn = listaJaccard(df_resultados_KeyBERT, df_asela=df_palabras_frecuencia[:len(df_asela)], df_psiq=df_palabras_frecuencia[:len(df_psiq)], df_EmoPro=df_palabras_frecuencia, df_EmoPro_altas=df_palabras_frecuencia[:len(df_EmoPro_altas)])
puntuaciones_GPT_topn = listaJaccard(df_resultados_GPT, df_asela=df_palabras_frecuencia[:len(df_asela)], df_psiq=df_palabras_frecuencia[:len(df_psiq)], df_EmoPro=df_palabras_frecuencia, df_EmoPro_altas=df_palabras_frecuencia[:len(df_EmoPro_altas)])


In [49]:
puntuaciones_textRank_topn

[('Corregido_TextRank_polaridad_negativas_EmoPro', 0.08155339805825243),
 ('Corregido_TextRank_etiqueta_negativas_EmoPro_alta', 0.16701902748414377),
 ('Corregido_TextRank_etiqueta_negativas_psiq', 0.24509803921568626),
 ('Corregido_TextRank_etiqueta_positivas_EmoPro', 0.08308895405669599),
 ('Corregido_TextRank_polaridad_positivas_EmoPro', 0.07804878048780488),
 ('Corregido_TextRank_polaridad_negativas_EmoPro_alta', 0.1729957805907173),
 ('Corregido_TextRank_polaridad_negativas_psiq', 0.2706270627062706),
 ('Corregido_TextRank_todas _EmoPro_alta', 0.1752136752136752),
 ('Corregido_TextRank_etiqueta_positivas_EmoPro_alta', 0.18025751072961374),
 ('Corregido_TextRank_polaridad_positivas_EmoPro_alta', 0.16382978723404254),
 ('Corregido_TextRank_todas_psiq', 0.26755852842809363),
 ('Corregido_TextRank_todas_EmoPro', 0.07992202729044834),
 ('Corregido_TextRank_etiqueta_negativas_EmoPro', 0.07976653696498054),
 ('Corregido_TextRank_polaridad_positivas_psiq', 0.25752508361204013),
 ('Corregi

In [50]:
puntuaciones_fractalidad_topn

[('fractalidad_etiqueta0_asela', 0.009009009009009009),
 ('fractalidad_todos_EmoPro', 0.09784735812133072),
 ('fractalidad_negativas_resultados_asela', 0.0),
 ('fractalidad_etiqueta5_EmoPro_prueba1', 0.23007348784624082),
 ('fractalidad_etiqueta0_EmoPro_prueba1', 0.44502617801047123),
 ('fractalidad_etiqueta5_asela', 0.0),
 ('fractalidad_polaridad_neg_EmoPro', 0.2727814175104229),
 ('fractalidad_etiqueta0_psiq', 0.15296367112810708),
 ('fractalidad_polaridad_pos_EmoPro', 0.18337801608579088),
 ('fractalidad_todos_psiq', 0.01631321370309951),
 ('fractalidad_resultados_todas_asela', 0.0),
 ('fractalidad_positivas_polaridad_psiq', 0.011382113821138212),
 ('fractalidad_positivas_resultados_asela', 0.009009009009009009),
 ('fractalidad_etiqueta5_psiq', 0.019704433497536946),
 ('fractalidad_negativas_polaridad_psiq', 0.018032786885245903),
 ('fractalidad_todos_EmoPro_altas', 0.04741833508956796),
 ('fractalidad_polaridad_pos_EmoPro_altas', 0.12128146453089245),
 ('fractalidad_polaridad_neg_E

In [51]:
puntuaciones_KeyBERT_topn

[('Salida_Etiqueta0_KB_asela', 0.0),
 ('Salida_Negativas_Todos_KB_asela', 0.0),
 ('Salida_Etiqueta0_KB_psiq', 0.03177257525083612),
 ('Salida_Todos_KB_asela', 0.0),
 ('Salida_Todos_KB_psiq', 0.00487012987012987),
 ('Salida_UnoporUno_KB_EmoPro', 0.14678423236514523),
 ('Salida_Etiqueta5_KB_psiq', 0.006568144499178982),
 ('Salida_Positivas_KB_psiq', 0.006644518272425249),
 ('Salida_Negativas_KB_EmoPro', 0.06858513189448441),
 ('Salida_UnoporUno_KB_final_psiq', 0.10989010989010989),
 ('Salida_Etiqueta5_KB_EmoPro', 0.0667948101874099),
 ('Salida_Etiqueta5_KB_asela', 0.0),
 ('Salida_Positivas_KB_EmoPro', 0.04973945997157745),
 ('Salida_Positivas_Todos_KB_asela', 0.0),
 ('Salida_Etiqueta0_KB_EmoPro', 0.16243386243386243),
 ('Salida_KB_Todos_EmoPro', 0.033718244803695153),
 ('Salida_Negativas_KB_psiq', 0.013245033112582781),
 ('Salida_UnoporUno_KB_final_asela', 0.0673076923076923),
 ('Salida_UnoporUno_KB_EmoPro_altas', 0.1260115606936416),
 ('Salida_Negativas_KB_EmoPro_altas', 0.0260960334029

In [52]:
puntuaciones_GPT_topn

[('palabras_prototipicas_100_EmoPro', 0.03648269410664172),
 ('palabras_emocionales_GPT_EmoPro_alta', 0.06614785992217899),
 ('palabras_emocionales_prompt_proto_EmoPro_alta', 0.03571428571428571),
 ('palabras_emocionales_GPT_psiq', 0.06497175141242938),
 ('palabras_emocionales_prompt_proto_psiq', 0.02981029810298103),
 ('palabras_prototipicas_100_asela', 0.028368794326241134),
 ('palabras_emocionales_prompt_proto_asela', 0.02097902097902098),
 ('palabras_prototipicas_100_EmoPro_alta', 0.05363984674329502),
 ('palabras_resultados_GPT_textos_unidos_EmoPro_alta', 0.04665314401622718),
 ('palabras_emocionales_prompt_proto_EmoPro', 0.03451492537313433),
 ('palabras_resultados_GPT_textos_unidos_psiq', 0.051829268292682924),
 ('palabras_emocionales_GPT_EmoPro', 0.050332383665717),
 ('palabras_emocionales_GPT_asela', 0.043795620437956206),
 ('palabras_resultados_GPT_textos_unidos_asela', 0.04716981132075472),
 ('palabras_resultados_GPT_textos_unidos_EmoPro', 0.029721955896452542),
 ('palabras_

### Medida de Jaccard entre Palabras aleatorias del texto y los algoritmos

In [53]:
def jaccardRandomVsAlgos(df, texto):
    comparacion = []

    for k, v in df.items():
        muestra = pd.DataFrame(randomSelect(len(v), texto), columns=['word'])
        muestra = lematizador(muestra, 'word')
        puntaje = jaccard(set(muestra['Lemma']), set(v['Lemma']))
        comparacion.append((k, puntaje))
    return comparacion

In [54]:
comparacion_TextRank_random = jaccardRandomVsAlgos(df_resultados_TextRank, texto)
comparacion_Fractalidad_random = jaccardRandomVsAlgos(df_resultados_fractalidad, texto)
comparacion_KeyBERT_random = jaccardRandomVsAlgos(df_resultados_KeyBERT, texto)
comparacion_GPT_random = jaccardRandomVsAlgos(df_resultados_GPT, texto)

In [55]:
comparacion_TextRank_random

[('Corregido_TextRank_polaridad_negativas_EmoPro', 0.18902439024390244),
 ('Corregido_TextRank_etiqueta_negativas_EmoPro_alta', 0.1488095238095238),
 ('Corregido_TextRank_etiqueta_negativas_psiq', 0.12280701754385964),
 ('Corregido_TextRank_etiqueta_positivas_EmoPro', 0.1656441717791411),
 ('Corregido_TextRank_polaridad_positivas_EmoPro', 0.13253012048192772),
 ('Corregido_TextRank_polaridad_negativas_EmoPro_alta', 0.12),
 ('Corregido_TextRank_polaridad_negativas_psiq', 0.1286549707602339),
 ('Corregido_TextRank_todas _EmoPro_alta', 0.15151515151515152),
 ('Corregido_TextRank_etiqueta_positivas_EmoPro_alta', 0.12352941176470589),
 ('Corregido_TextRank_polaridad_positivas_EmoPro_alta', 0.13333333333333333),
 ('Corregido_TextRank_todas_psiq', 0.11834319526627218),
 ('Corregido_TextRank_todas_EmoPro', 0.15950920245398773),
 ('Corregido_TextRank_etiqueta_negativas_EmoPro', 0.16363636363636364),
 ('Corregido_TextRank_polaridad_positivas_psiq', 0.14906832298136646),
 ('Corregido_TextRank_eti

In [56]:
comparacion_Fractalidad_random

[('fractalidad_etiqueta0_asela', 0.017857142857142856),
 ('fractalidad_todos_EmoPro', 0.11303511303511303),
 ('fractalidad_negativas_resultados_asela', 0.0),
 ('fractalidad_etiqueta5_EmoPro_prueba1', 0.18842105263157893),
 ('fractalidad_etiqueta0_EmoPro_prueba1', 0.2722335369993211),
 ('fractalidad_etiqueta5_asela', 0.0),
 ('fractalidad_polaridad_neg_EmoPro', 0.1891025641025641),
 ('fractalidad_etiqueta0_psiq', 0.10471204188481675),
 ('fractalidad_polaridad_pos_EmoPro', 0.16109422492401215),
 ('fractalidad_todos_psiq', 0.022082018927444796),
 ('fractalidad_resultados_todas_asela', 0.0),
 ('fractalidad_positivas_polaridad_psiq', 0.0189873417721519),
 ('fractalidad_positivas_resultados_asela', 0.0),
 ('fractalidad_etiqueta5_psiq', 0.01889763779527559),
 ('fractalidad_negativas_polaridad_psiq', 0.0304),
 ('fractalidad_todos_EmoPro_altas', 0.07291666666666667),
 ('fractalidad_polaridad_pos_EmoPro_altas', 0.12280701754385964),
 ('fractalidad_polaridad_neg_EmoPro_altas', 0.14369158878504673)

In [57]:
comparacion_KeyBERT_random

[('Salida_Etiqueta0_KB_asela', 0.0),
 ('Salida_Negativas_Todos_KB_asela', 0.009009009009009009),
 ('Salida_Etiqueta0_KB_psiq', 0.027243589743589744),
 ('Salida_Todos_KB_asela', 0.0),
 ('Salida_Todos_KB_psiq', 0.007751937984496124),
 ('Salida_UnoporUno_KB_EmoPro', 0.12011863568956994),
 ('Salida_Etiqueta5_KB_psiq', 0.017488076311605722),
 ('Salida_Positivas_KB_psiq', 0.012678288431061807),
 ('Salida_Negativas_KB_EmoPro', 0.06939338235294118),
 ('Salida_UnoporUno_KB_final_psiq', 0.07521367521367521),
 ('Salida_Etiqueta5_KB_EmoPro', 0.06896551724137931),
 ('Salida_Etiqueta5_KB_asela', 0.00909090909090909),
 ('Salida_Positivas_KB_EmoPro', 0.055),
 ('Salida_Positivas_Todos_KB_asela', 0.01834862385321101),
 ('Salida_Etiqueta0_KB_EmoPro', 0.12272055199605716),
 ('Salida_KB_Todos_EmoPro', 0.047064096817570594),
 ('Salida_Negativas_KB_psiq', 0.01282051282051282),
 ('Salida_UnoporUno_KB_final_asela', 0.027522935779816515),
 ('Salida_UnoporUno_KB_EmoPro_altas', 0.09472551130247578),
 ('Salida_Neg

In [58]:
comparacion_GPT_random

[('palabras_prototipicas_100_EmoPro', 0.029940119760479042),
 ('palabras_emocionales_GPT_EmoPro_alta', 0.028409090909090908),
 ('palabras_emocionales_prompt_proto_EmoPro_alta', 0.010752688172043012),
 ('palabras_emocionales_GPT_psiq', 0.040229885057471264),
 ('palabras_emocionales_prompt_proto_psiq', 0.016129032258064516),
 ('palabras_prototipicas_100_asela', 0.02857142857142857),
 ('palabras_emocionales_prompt_proto_asela', 0.005376344086021506),
 ('palabras_prototipicas_100_EmoPro_alta', 0.04093567251461988),
 ('palabras_resultados_GPT_textos_unidos_EmoPro_alta', 0.02727272727272727),
 ('palabras_emocionales_prompt_proto_EmoPro', 0.021621621621621623),
 ('palabras_resultados_GPT_textos_unidos_psiq', 0.02727272727272727),
 ('palabras_emocionales_GPT_EmoPro', 0.022727272727272728),
 ('palabras_emocionales_GPT_asela', 0.034482758620689655),
 ('palabras_resultados_GPT_textos_unidos_asela', 0.009009009009009009),
 ('palabras_resultados_GPT_textos_unidos_EmoPro', 0.037037037037037035),
 ('